# **UNIVERSIDADE FEDERAL DO CEARÁ**
---
Disciplina: Introdução à análise em Big Data

---

Professor: Luiz Alexandre

---

Alunos:
1.   Júlio César Gama Feitosa Freitas - 583956
2.   Vitória Freire Rocha Teixeira de Oliveira - 587661

---
Data: 13/09/2026

# 🧪 Lab 6 — Silver: enriquecer com contexto

## 🎯 Objetivo

Construir a camada Silver: **juntar** transações com clientes (JOIN) e **derivar** colunas de análise (data, dia da semana, faixa de valor). É aqui que o dado ganha contexto de negócio — cada transação passa a saber de que segmento ela é.

O resultado, `silver_transactions`, é exatamente o que a Gold (Lab 7) vai agregar.

**Rota B — DuckDB + Python/Google Colab**

## Configuração Inicial

O Lab 6 utiliza os Parquets gerados no Lab 5:

- `bigdata/bronze/customers.parquet`
- `bigdata/bronze/transactions.parquet`

Também criaremos a pasta `bigdata/silver/`.

In [4]:
# Importa as bibliotecas e cria as pastas utilizadas pelo laboratório.
import os
import shutil
import duckdb

os.makedirs("bigdata/raw/customers", exist_ok=True)
os.makedirs("bigdata/raw/transactions", exist_ok=True)
os.makedirs("bigdata/bronze", exist_ok=True)
os.makedirs("bigdata/silver", exist_ok=True)

# Abre uma conexão DuckDB.
con = duckdb.connect()

print("Ambiente preparado.")

Ambiente preparado.


In [5]:
uploaded = [
    "../customers_synthetic.csv",
    "../transactions_synthetic.csv",
    "../fraud_labels.csv"
]

missing = [f for f in uploaded if not os.path.exists(f)] # Verifica se todos os arquivos necessários foram carregados
if missing:
    raise FileNotFoundError("Arquivos ausentes: " + ", ".join(missing))

print("\n✓ Os 3 datasets foram encontrados.")


✓ Os 3 datasets foram encontrados.


In [6]:
# Copia os CSVs para a estrutura Raw do projeto.
for name in uploaded:
    if name == "customers_synthetic.csv":
        shutil.copy(name, "bigdata/raw/customers/customers_synthetic.csv")
    elif name == "transactions_synthetic.csv":
        shutil.copy(name, "bigdata/raw/transactions/transactions_synthetic.csv")

print("Arquivos Raw preparados.")

Arquivos Raw preparados.


## Passo 1 — Carregar a Bronze e montar a Silver

In [7]:
# Define os caminhos dos arquivos Raw.
customers_raw = "bigdata/raw/customers/customers_synthetic.csv"
transactions_raw = "bigdata/raw/transactions/transactions_synthetic.csv"

# Recria a Bronze de clientes com as mesmas regras utilizadas no Lab 5.
con.sql(f"""
CREATE OR REPLACE TABLE bronze_customers AS
SELECT DISTINCT
    customer_id, name, cpf, email, segment,
    CAST(credit_score AS INT) AS credit_score,
    CAST(created_at AS DATE) AS created_at
FROM read_csv_auto('{customers_raw}')
WHERE customer_id IS NOT NULL
  AND credit_score BETWEEN 300 AND 900
""")

# Recria a Bronze de transações com as mesmas regras utilizadas no Lab 5.
# O CASE converte explicitamente True/False para BOOLEAN.
con.sql(f"""
CREATE OR REPLACE TABLE bronze_transactions AS
SELECT DISTINCT
    transaction_id, customer_id,
    CAST(amount AS FLOAT) AS amount,
    transaction_type, status,
    CAST(risk_score AS FLOAT) AS risk_score,
    CASE
        WHEN is_fraud = 'True' THEN true
        ELSE false
    END AS is_fraud,
    CAST(timestamp AS TIMESTAMP) AS ts
FROM read_csv_auto('{transactions_raw}')
WHERE amount > 0
  AND customer_id IS NOT NULL
""")

# Confere as quantidades reconstruídas na Bronze.
con.sql("SELECT COUNT(*) AS total FROM bronze_customers").show()
con.sql("SELECT COUNT(*) AS total FROM bronze_transactions").show()

┌───────┐
│ total │
│ int64 │
├───────┤
│  9993 │
└───────┘

┌────────┐
│ total  │
│ int64  │
├────────┤
│ 100000 │
└────────┘



In [8]:
# Salva a Bronze reconstruída em Parquet.
con.sql("""
COPY bronze_customers
TO 'bigdata/bronze/customers.parquet'
(FORMAT PARQUET)
""")

con.sql("""
COPY bronze_transactions
TO 'bigdata/bronze/transactions.parquet'
(FORMAT PARQUET)
""")

print("Bronze layer salva em bigdata/bronze/")

Bronze layer salva em bigdata/bronze/


In [9]:
# Carrega a Bronze salva em Parquet nesta sessão.
con.sql("CREATE OR REPLACE TABLE bronze_customers AS SELECT * FROM read_parquet('bigdata/bronze/customers.parquet')")
con.sql("CREATE OR REPLACE TABLE bronze_transactions AS SELECT * FROM read_parquet('bigdata/bronze/transactions.parquet')")

# Cria a Silver juntando transações com clientes e derivando as colunas solicitadas.
con.sql("""
CREATE OR REPLACE TABLE silver_transactions AS
SELECT
  t.transaction_id, t.customer_id, t.amount, t.transaction_type,
  t.status, t.risk_score, t.is_fraud, t.ts,
  c.segment, c.credit_score,
  year(t.ts)  AS year,
  month(t.ts) AS month,
  day(t.ts)   AS day,
  dayofweek(t.ts) AS day_of_week,
  CASE
    WHEN t.amount < 100  THEN 'baixo'
    WHEN t.amount < 1000 THEN 'medio'
    ELSE 'alto'
  END AS amount_band
FROM bronze_transactions t
JOIN bronze_customers c ON t.customer_id = c.customer_id
""")

# Confere a quantidade de registros criada na Silver.
con.sql("SELECT COUNT(*) FROM silver_transactions").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       100000 │
└──────────────┘



## Passo 2 — Validar o JOIN

In [10]:
# Compara a quantidade de registros da Silver com a Bronze.
con.sql("SELECT COUNT(*) AS silver, (SELECT COUNT(*) FROM bronze_transactions) AS bronze FROM silver_transactions").show()

# Verifica se alguma linha ficou sem segmento.
con.sql("SELECT COUNT(*) AS sem_segmento FROM silver_transactions WHERE segment IS NULL").show()

┌────────┬────────┐
│ silver │ bronze │
│ int64  │ int64  │
├────────┼────────┤
│ 100000 │ 100000 │
└────────┴────────┘

┌──────────────┐
│ sem_segmento │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘



In [11]:
# Identifica customer_id presentes nas transações, mas ausentes na tabela de clientes.
con.sql("""
SELECT DISTINCT t.customer_id
FROM bronze_transactions t
LEFT JOIN bronze_customers c ON t.customer_id = c.customer_id
WHERE c.customer_id IS NULL
""").show()

┌─────────────┐
│ customer_id │
│    int64    │
└─────────────┘
    0 rows   



In [12]:
# Consulta a origem Raw para verificar os IDs órfãos.
con.sql("""
SELECT customer_id
FROM read_csv_auto('bigdata/raw/customers/customers_synthetic.csv')
WHERE customer_id IN (
  SELECT DISTINCT t.customer_id
  FROM bronze_transactions t
  LEFT JOIN bronze_customers c ON t.customer_id = c.customer_id
  WHERE c.customer_id IS NULL
)
""").show()

┌─────────────┐
│ customer_id │
│    int64    │
└─────────────┘
    0 rows   



## Passo 3 — Prévia do cruzamento por segmento

In [13]:
# Confirma que nenhuma transação da Silver ficou sem segmento.
con.sql("SELECT COUNT(*) FROM silver_transactions WHERE segment IS NULL").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘



In [14]:
# Exibe as colunas derivadas para conferir se os valores estão coerentes.
con.sql("""
SELECT amount, amount_band, year, month, day, day_of_week
FROM silver_transactions
LIMIT 10
""").show()

┌───────────┬─────────────┬───────┬───────┬───────┬─────────────┐
│  amount   │ amount_band │ year  │ month │  day  │ day_of_week │
│   float   │   varchar   │ int64 │ int64 │ int64 │    int64    │
├───────────┼─────────────┼───────┼───────┼───────┼─────────────┤
│  82.37801 │ baixo       │  2024 │    12 │    23 │           1 │
│ 229.38356 │ medio       │  2024 │    12 │    23 │           1 │
│  97.73424 │ baixo       │  2024 │    12 │    23 │           1 │
│ 14.574973 │ baixo       │  2024 │    12 │    23 │           1 │
│ 34.327923 │ baixo       │  2024 │    12 │    23 │           1 │
│ 1488.2599 │ alto        │  2024 │    12 │    24 │           2 │
│ 284.52344 │ medio       │  2024 │    12 │    24 │           2 │
│  31.73015 │ baixo       │  2024 │    12 │    24 │           2 │
│  79.49905 │ baixo       │  2024 │    12 │    24 │           2 │
│  66.34288 │ baixo       │  2024 │    12 │    24 │           2 │
└───────────┴─────────────┴───────┴───────┴───────┴─────────────┘
  10 rows 

## Prévia da Gold: fraude por segmento

In [15]:
# Calcula a quantidade de transações e a taxa de fraude por segmento.
# A Gold vai formalizar essa análise no Lab 7.
con.sql("""
SELECT segment,
       COUNT(*) AS transacoes,
       ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 2) AS taxa_fraude_pct
FROM silver_transactions
GROUP BY segment
ORDER BY taxa_fraude_pct DESC
""").show()

┌───────────┬────────────┬─────────────────┐
│  segment  │ transacoes │ taxa_fraude_pct │
│  varchar  │   int64    │     double      │
├───────────┼────────────┼─────────────────┤
│ High-Risk │       9155 │             7.7 │
│ Standard  │      29689 │            2.21 │
│ Premium   │      61156 │            0.77 │
└───────────┴────────────┴─────────────────┘



## Passo 4 — Persistir a Silver (o Lab 7 e o Lab 10 leem daqui)

In [16]:
# Salva a Silver em Parquet para ser utilizada pelos próximos laboratórios.
con.sql("COPY silver_transactions TO 'bigdata/silver/transactions_enriched.parquet' (FORMAT PARQUET)")
print("Silver layer salva em bigdata/silver/transactions_enriched.parquet")

Silver layer salva em bigdata/silver/transactions_enriched.parquet


## ✅ Checkpoint

- [ ] `silver_transactions` criada a partir do JOIN transações × clientes
- [ ] Contagem da Silver bate com a `bronze_transactions` — ou a diferença é pequena e explicada por clientes ausentes na origem
- [ ] Nenhuma linha com `segment IS NULL`
- [ ] Colunas derivadas `year/month/day`, `day_of_week` e `amount_band` presentes
- [ ] `bigdata/silver/transactions_enriched.parquet` salvo

---